# ORF v4.0.0 — One-Cell Runner

Run the cell below to go **from a project `.zip` to execution** in a single step:

1. **Get the project** — if `main.py` isn't already sitting next to this notebook, a real **upload button** appears (Colab's native file picker, or an OS file dialog in local Jupyter) so you pick the exact `.zip` you mean. Directory auto-scan is only a fallback for headless/non-interactive runs.
2. **Bootstrap Ollama if needed** — if a configured endpoint points at `localhost` and nothing answers, and this is Colab, Ollama is installed and started *inside the Colab VM* and the required models are pulled automatically. (In Colab, `localhost` is the cloud VM Colab gave you, not your laptop — there's no tunnel to your machine, so the model server has to actually run inside the notebook's runtime.)
3. **Install** the minimal Python dependencies.
4. **Configure** the target from the `CONFIG` block. **Defaults to local Ollama for both attacker and target** — no tokens, no remote host, nothing to fill in. Point `TARGET_PROVIDER` at `sse`/`openai` instead if you're auditing a remote endpoint.
5. **Execute** `main.py` (`probe` smoke test by default, or the full `run` pipeline).

A `PERFORMANCE` block controls concurrency. `main.py` v4.0.0 rate-limits only the actual network round trips to the attacker/target LLMs — local work (whitebox scanning, payload composition, SAST evaluation) runs fully in parallel across categories — so raising `MAX_CONCURRENT_TASKS` speeds up the run without touching a GPU or changing what gets sent to the target.

**Before running:** with the Ollama default, Colab handles setup itself (step 2 above). Running locally instead, just make sure `ollama serve` is running and `ollama pull dolphin-llama3` + `ollama pull llama3.2` have been run. Switching `TARGET_PROVIDER` to `sse`/`openai` additionally needs `TARGET_API_KEY` filled in.

---

### Embedding-lab mode (`MODE = "vvs"`)
Set `MODE = "vvs"` to run the `vvs_test.py` harness against the embedding-attack-lab boundary service instead of the LLM pipeline. It ignores the Ollama/attacker/judge and SSE-target settings and never bootstraps a model — it just drives the HTTP attacks from `TESTING.md` against `VVS_BASE_URL` (default `https://your-target.example`). Pick the attack with `VVS_CMD` (`smoke` / `kplus` / `attack-a`–`attack-d` / `boundary` / `all`).


In [ ]:
# =====================================================================
# ORF v4.0.0 — ONE-CELL RUNNER:  upload/unzip -> bootstrap ollama -> install ->
#                            configure -> execute
# Edit the CONFIG block below, then Run this cell. Works in Colab or
# local Jupyter. If the project is already unzipped next to this
# notebook, the upload/unzip step is skipped automatically.
# =====================================================================
import os, sys, glob, zipfile, subprocess, shutil, time
from pathlib import Path

# ------------------------------- CONFIG -------------------------------
ZIP_PATH      = ""     # path to the project .zip; "" = use the upload button / auto-detect
UPLOAD_BUTTON = True   # show a click-to-upload button when no project is found nearby
MODE          = "probe"     # "probe" = connectivity smoke test | "run" = full red-team pipeline | "vvs" = embedding-lab harness (TESTING.md)

# --- VVS EMBEDDING-LAB TARGET (only used when MODE="vvs") --------------
# Runs the vvs_test.py harness against the embedding-attack-lab boundary
# service from TESTING.md (default: the live your-target.example deployment). This
# mode is self-contained HTTP: it ignores every Ollama / attacker / judge and
# SSE-target setting in this cell and never bootstraps a local model.
VVS_BASE_URL = "https://your-target.example"   # embedding-lab base URL
VVS_CMD      = "all"    # smoke | kplus | attack-a | attack-b | attack-c | attack-d | boundary | all
VVS_ARMS     = "A,B,C,D"                  # storage arms to exercise
VVS_DIM      = 2560                       # M1 = Qwen3-Embedding-4B (embedding dim)
VVS_DEBUG    = False                      # log each HTTP request to stderr

# --- TARGET: defaults to local Ollama (no tokens, no remote host needed) ----
# Switch to "sse" (generic SSE chatbot) or "openai" (OpenAI-compatible /v1) to
# audit a remote endpoint instead -- then fill in TARGET_BASE_URL/API_KEY below.
TARGET_PROVIDER  = "ollama"                    # ollama (native) | sse | openai
TARGET_BASE_URL  = "http://localhost:11434"    # Ollama host (ollama) | full chat URL (sse) | /v1 base (openai)
TARGET_API_KEY   = ""     # not needed for local Ollama; set if your target requires a Bearer token
TARGET_COOKIE    = ""     # optional
SECONDARY_MODEL  = "llama3.2"   # target model id -- must already be pulled: `ollama pull llama3.2`
PROBE_PROMPT     = "Connectivity smoke test - please reply with a short confirmation."

# --- OPTIONAL: indirect prompt injection via file upload (sse only) ----
# Leave blank/False to use the direct (in-query) delivery path.
TARGET_FILE_UPLOAD_URL = ""     # e.g. https://your-chat-host.example/api/file/upload?type=chat
INDIRECT_MODE          = ""     # "" = direct | "file_upload" = upload doc then attach it
FILE_ATTACH_MODE       = "body_field"   # body_field | session | query_ref
FILE_ATTACH_FIELD      = "file_ids"

# --- ATTACKER (only used when MODE='run'; needs a reachable LLM) -------
ATTACKER_PROVIDER = "ollama"                  # 'ollama' (local) or 'openai'
ATTACKER_BASE_URL = "http://localhost:11434"  # or an OpenAI-compatible /v1 base URL
ATTACKER_API_KEY  = ""
ATTACKER_MODEL    = "dolphin-llama3"    # must already be pulled: `ollama pull dolphin-llama3`

# --- MULTI-TECHNIQUE COMPOSITION (MODE='run' only) ----------------------
# main.py defaults ORF_COMPOSE_MODE to "off" (legacy single delivery-framing
# wrapper only -- no obfuscation/structural/framing/multi-turn techniques).
# "auto" turns on the full technique composer: surface-aware selection from
# url_fragmentation, image_markdown_zero_click, multimodal_image_injection,
# multi_turn_logic_injection, many_shot, cot_manipulation,
# multi_turn_context_poisoning, memory_poisoning, multilingual_injection,
# payload_splitting, crescendo (see `python main.py techniques`).
# "manual" uses INJECTION_TECHNIQUES verbatim instead of auto-selecting.
COMPOSE_MODE          = "auto"    # off | auto | manual
INJECTION_TECHNIQUES  = ""        # only used when COMPOSE_MODE="manual", e.g.
                                   # "base64,image_markdown_zero_click,cot_manipulation,memory_poisoning"
EXFIL_HOST            = ""        # host used by exfil-shaped structural techniques (url_fragmentation /
                                   # image markdown); "" = tool default (collector.redteam-lab.example)

# --- AGENTIC SUPERDATA-EXTRACTOR (optional; MODE='run' only) ------------
# Lets the attacker LLM request grounded intel mid-generation via a
# NEED_INTEL: line (answered from a frontend/backend extraction pass run
# once before the audit starts). Leave SUPERDATA_SOURCE/URL blank to skip
# the extraction pass entirely -- AGENTIC_INTEL then has nothing to draw on.
AGENTIC_INTEL     = False   # let the attacker ask the Superdata bridge for intel mid-generation
SUPERDATA_SOURCE  = ""      # local path to the target's frontend codebase (optional)
SUPERDATA_URL     = ""      # or a live URL to scrape instead (optional)
SUPERDATA_REFRESH = False   # force re-extraction even if intel already exists

# --- OLLAMA BOOTSTRAP (Colab only) --------------------------------------
# If a *_PROVIDER above is "ollama" and its *_BASE_URL is localhost/127.0.0.1
# and unreachable, auto-install + start Ollama inside this Colab VM and pull
# the models this run needs. Colab's "localhost" is the cloud VM Colab gave
# you, not your laptop -- there's no tunnel to your machine, so the model
# server has to actually run inside the notebook's runtime. Has no effect
# outside Colab or against a non-local TARGET_BASE_URL/ATTACKER_BASE_URL.
AUTO_BOOTSTRAP_OLLAMA = True

# --- PERFORMANCE (CPU-only; no GPU switch, no quality tradeoff) --------
# Caps concurrent network calls to the attacker/target LLMs. Local work
# (whitebox scan, payload composition, SAST) is unthrottled regardless.
# Raise this only as far as the target endpoint can tolerate -- it is a
# real request-rate increase against whatever TARGET_BASE_URL points to.
# 1 = original conservative default. 3-4 is a reasonable starting point
# for a target you own/control; keep it at 1 for unfamiliar third-party
# endpoints so you don't trip their rate limiting.
MAX_CONCURRENT_TASKS = 1

# Deps main.py actually needs (siblings are stdlib-only). 'datasets' is imported
# but unused by the runner -> stubbed if absent to skip its heavy install.
DEPS = ["aiohttp", "requests", "python-dotenv", "sqlalchemy", "pydantic"]
# ------------------------------------------------------------------------

def sh(msg): print(f"\033[1;36m>> {msg}\033[0m")

t0 = time.time()

# 1) locate project (upload/unzip only if main.py isn't already present) -
here = Path.cwd()

def find_main(root):
    if (root / "main.py").exists():
        return root
    hits = list(root.rglob("main.py"))
    return hits[0].parent if hits else None

def find_project_upwards(start, max_up=4):
    # This notebook may live in a subfolder (e.g. colab/); main.py + data/ then
    # sit in an ancestor, not a descendant. Check this dir, then walk UP.
    start = Path(start)
    for base in [start, *list(start.parents)[:max_up]]:
        if (base / "main.py").exists():
            return base
    return None

def find_zip_candidates():
    cands = []
    if ZIP_PATH and Path(ZIP_PATH).exists():
        cands.append(Path(ZIP_PATH))
    for base in [here, here.parent, Path("/content"), Path.home() / "Downloads"]:
        if base.exists():
            cands += [Path(p) for p in sorted(glob.glob(str(base / "*.zip")))]
    seen, out = set(), []
    for c in cands:
        rc = c.resolve()
        if rc not in seen:
            seen.add(rc)
            out.append(c)
    return out

def _bad_zip_hint(path):
    size = path.stat().st_size if path.exists() else 0
    if size == 0:
        return " (0 bytes -- the upload/download never completed)"
    head = path.open("rb").read(64)
    if head.lstrip().lower().startswith((b"<!doctype", b"<html")):
        return (" (this is an HTML page, not a zip -- typical of a redirected/interstitial "
                 "download link, e.g. an unconfirmed Google Drive share link instead of the "
                 "actual file)")
    return f" (first bytes: {head[:24]!r})"

def prompt_upload():
    """Show a real click-to-upload control and return the picked file's path, or
    None if unavailable/cancelled (e.g. a headless/non-interactive run)."""
    try:
        from google.colab import files as colab_files
    except ImportError:
        colab_files = None

    if colab_files is not None:
        sh("Click 'Choose Files' below and select your project .zip ...")
        uploaded = colab_files.upload()
        if not uploaded:
            return None
        name = next(iter(uploaded))
        dest = here / name
        dest.write_bytes(uploaded[name])
        return dest

    # Local Jupyter / plain Python: fall back to a native OS file-picker dialog.
    try:
        import tkinter as tk
        from tkinter import filedialog
        root_win = tk.Tk()
        root_win.withdraw()
        root_win.attributes("-topmost", True)
        sh("Opening a file picker -- select your project .zip ...")
        path = filedialog.askopenfilename(
            title="Select the project .zip",
            filetypes=[("Zip files", "*.zip"), ("All files", "*.*")],
        )
        root_win.destroy()
        return Path(path) if path else None
    except Exception:
        return None

project = find_project_upwards(here) or find_main(here)
if project is None:
    z = None
    if ZIP_PATH and Path(ZIP_PATH).exists() and zipfile.is_zipfile(Path(ZIP_PATH)):
        z = Path(ZIP_PATH)
    elif UPLOAD_BUTTON:
        picked = prompt_upload()
        if picked is not None:
            if zipfile.is_zipfile(picked):
                z = picked
            else:
                raise SystemExit(
                    f"Uploaded/selected file {picked} is not a valid zip.{_bad_zip_hint(picked)}"
                )

    if z is None:
        # Fallback for headless/non-interactive runs: auto-scan for a *.zip nearby.
        candidates = find_zip_candidates()
        if not candidates:
            raise SystemExit(
                "No main.py, no uploaded .zip, and no .zip nearby. "
                "Set ZIP_PATH in CONFIG, or enable UPLOAD_BUTTON and re-run interactively."
            )
        bad = []
        for cand in candidates:
            if zipfile.is_zipfile(cand):
                z = cand
                break
            bad.append(cand)
        if z is None:
            lines = ["No valid .zip found among candidates:"]
            lines += [f"  - {b}{_bad_zip_hint(b)}" for b in bad]
            lines.append(
                "Re-download/re-upload the .zip (verify it opens locally first), "
                "or set ZIP_PATH to a file you've confirmed is good."
            )
            raise SystemExit("\n".join(lines))
        if len(candidates) > 1:
            sh(f"Found {len(candidates)} *.zip candidate(s); using the first valid one: {z}")

    sh(f"Unzipping {z} ...")
    dest = here / "orf_unzipped"
    if dest.exists():
        shutil.rmtree(dest)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(dest)
    project = find_main(dest)
    if project is None:
        raise SystemExit(f"Unzipped {z} but found no main.py inside it.")
else:
    sh("main.py already present - skipping upload/unzip.")
sh(f"Project root: {project}")
os.chdir(project)

# 1b) bootstrap a local Ollama server if this run needs one and it's unreachable
def _is_local_host(url):
    from urllib.parse import urlparse
    return (urlparse(url).hostname or "").lower() in ("localhost", "127.0.0.1", "::1", "")

def _ollama_reachable(base_url, timeout=3):
    import urllib.request
    try:
        with urllib.request.urlopen(f"{base_url.rstrip('/')}/api/tags", timeout=timeout):
            return True
    except Exception:
        return False

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def _find_ollama_binary():
    found = shutil.which("ollama")
    if found:
        return found
    for cand in ("/usr/local/bin/ollama", "/usr/bin/ollama",
                 str(Path.home() / ".ollama" / "bin" / "ollama")):
        if Path(cand).exists():
            return cand
    return None

def _ensure_zstd():
    """The official Ollama installer needs 'zstd' to unpack its archive; Colab's
    base image doesn't ship it. Install it via apt (Colab runs as root, no sudo)."""
    if shutil.which("zstd"):
        return True
    sh("Installing 'zstd' (required by the Ollama installer) ...")
    env = {**os.environ, "DEBIAN_FRONTEND": "noninteractive"}
    r = subprocess.run(["apt-get", "install", "-y", "-qq", "zstd"],
                        capture_output=True, text=True, env=env)
    if r.returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True, text=True, env=env)
        r = subprocess.run(["apt-get", "install", "-y", "-qq", "zstd"],
                            capture_output=True, text=True, env=env)
    if r.returncode != 0:
        print(r.stdout[-1500:])
        print(r.stderr[-1500:])
    return shutil.which("zstd") is not None

def ensure_ollama_ready(base_url, models):
    if not AUTO_BOOTSTRAP_OLLAMA or not _is_local_host(base_url) or _ollama_reachable(base_url):
        return
    if not _in_colab():
        sh(f"!! {base_url} is unreachable and this isn't Colab -- start it yourself "
           f"('ollama serve') or point the *_BASE_URL at a reachable host.")
        return

    ollama_bin = _find_ollama_binary()
    if ollama_bin is None:
        if not _ensure_zstd():
            sh("!! Could not install 'zstd' via apt-get -- the Ollama installer needs it. "
               "Try running `!apt-get install -y zstd` in its own cell first.")
        sh(f"{base_url} unreachable -- installing Ollama inside this Colab VM ...")
        install = subprocess.run(
            "curl -fsSL https://ollama.com/install.sh | sh",
            shell=True, capture_output=True, text=True,
        )
        # Show the installer's own output -- swallowing this is exactly how a failed
        # install turns into a confusing 'ollama: No such file or directory' later.
        print(install.stdout[-3000:])
        if install.returncode != 0:
            print(install.stderr[-3000:])
        ollama_bin = _find_ollama_binary()
        if ollama_bin is None:
            raise SystemExit(
                f"Ollama installer exited {install.returncode} and no 'ollama' binary was found "
                "afterward (checked PATH, /usr/local/bin, /usr/bin). See the installer output "
                "above for the real cause -- most often a blocked/failed download inside this "
                "Colab VM. You can also try installing manually in a separate cell first:\n"
                "  !curl -fsSL https://ollama.com/install.sh | sh"
            )
    else:
        sh(f"Found existing Ollama binary at {ollama_bin}.")

    sh("Starting 'ollama serve' in the background ...")
    subprocess.Popen([ollama_bin, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(30):
        if _ollama_reachable(base_url, timeout=2):
            break
        time.sleep(2)
    else:
        sh("!! Ollama still not answering after ~60s -- the probe/run below will likely fail.")
        return

    for model in models:
        if model:
            sh(f"Pulling model '{model}' (can take a few minutes depending on size) ...")
            subprocess.run([ollama_bin, "pull", model], check=False)

if AUTO_BOOTSTRAP_OLLAMA and MODE != "vvs":
    needed = {m for m in (ATTACKER_MODEL, SECONDARY_MODEL) if m}
    if ATTACKER_PROVIDER == "ollama":
        ensure_ollama_ready(ATTACKER_BASE_URL, needed)
    if TARGET_PROVIDER == "ollama" and TARGET_BASE_URL != ATTACKER_BASE_URL:
        ensure_ollama_ready(TARGET_BASE_URL, needed)

# 2) install the minimal dependency set ----------------------------------
sh("Installing dependencies (quiet) ...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *DEPS], check=False)

try:
    import datasets  # noqa: F401
    sh("'datasets' available.")
except Exception:
    shim = project / "_shims"
    shim.mkdir(exist_ok=True)
    (shim / "datasets.py").write_text(
        "def load_dataset(*a, **k):\n"
        "    raise RuntimeError('datasets stub - not used by the ORF v4.0.0 runner')\n"
    )
    os.environ["PYTHONPATH"] = f"{shim}{os.pathsep}" + os.environ.get("PYTHONPATH", "")
    sh("Stubbed the unused 'datasets' import (skipped its heavy install).")

# 3) write PLACEHOLDER.env (gitignored; ephemeral for this session) -----
env_lines = [
    f"TARGET_PROVIDER={TARGET_PROVIDER}",
    f"TARGET_BASE_URL={TARGET_BASE_URL}",
    f"TARGET_API_KEY={TARGET_API_KEY}",
    f"TARGET_COOKIE={TARGET_COOKIE}",
    f"OLLAMA_SECONDARY_MODEL={SECONDARY_MODEL}",
    f"ATTACKER_PROVIDER={ATTACKER_PROVIDER}",
    f"OLLAMA_BASE_URL={ATTACKER_BASE_URL}",
    f"OLLAMA_PRIMARY_MODEL={ATTACKER_MODEL}",
    f"OLLAMA_MAX_CONCURRENT_TASKS={MAX_CONCURRENT_TASKS}",
]
if TARGET_FILE_UPLOAD_URL:
    env_lines.append(f"TARGET_FILE_UPLOAD_URL={TARGET_FILE_UPLOAD_URL}")
if INDIRECT_MODE:
    env_lines.append(f"TARGET_INDIRECT_MODE={INDIRECT_MODE}")
    env_lines.append(f"TARGET_FILE_ATTACH_MODE={FILE_ATTACH_MODE}")
    env_lines.append(f"TARGET_FILE_ATTACH_FIELD={FILE_ATTACH_FIELD}")
(project / "PLACEHOLDER.env").write_text("\n".join(env_lines) + "\n")

# 4) build the command and execute (streaming output) --------------------
if MODE == "probe":
    cmd = [sys.executable, "main.py", "probe",
           "--target-provider", TARGET_PROVIDER,
           "--target-base-url", TARGET_BASE_URL,
           "--target-api-key", TARGET_API_KEY,
           "--secondary-model", SECONDARY_MODEL,
           "--prompt", PROBE_PROMPT]
    if TARGET_COOKIE:
        cmd += ["--target-cookie", TARGET_COOKIE]
    if INDIRECT_MODE == "file_upload":
        cmd += ["--indirect-mode", INDIRECT_MODE,
                "--target-file-upload-url", TARGET_FILE_UPLOAD_URL,
                "--file-attach-mode", FILE_ATTACH_MODE,
                "--file-attach-field", FILE_ATTACH_FIELD]
elif MODE == "run":
    cmd = [sys.executable, "main.py", "run",
           "--attacker-provider", ATTACKER_PROVIDER,
           "--attacker-base-url", ATTACKER_BASE_URL,
           "--model", ATTACKER_MODEL,
           "--target-provider", TARGET_PROVIDER,
           "--target-base-url", TARGET_BASE_URL,
           "--target-api-key", TARGET_API_KEY,
           "--secondary-model", SECONDARY_MODEL,
           "--compose", COMPOSE_MODE]
    if ATTACKER_API_KEY:
        cmd += ["--attacker-api-key", ATTACKER_API_KEY]
    if TARGET_COOKIE:
        cmd += ["--target-cookie", TARGET_COOKIE]
    if INDIRECT_MODE == "file_upload":
        cmd += ["--indirect-mode", INDIRECT_MODE,
                "--target-file-upload-url", TARGET_FILE_UPLOAD_URL,
                "--file-attach-mode", FILE_ATTACH_MODE,
                "--file-attach-field", FILE_ATTACH_FIELD]
    if COMPOSE_MODE == "manual" and INJECTION_TECHNIQUES:
        cmd += ["--injection-techniques", INJECTION_TECHNIQUES]
    if EXFIL_HOST:
        cmd += ["--exfil-host", EXFIL_HOST]
    if AGENTIC_INTEL:
        cmd += ["--agentic-intel"]
    if SUPERDATA_SOURCE:
        cmd += ["--superdata-source", SUPERDATA_SOURCE]
    if SUPERDATA_URL:
        cmd += ["--superdata-url", SUPERDATA_URL]
    if SUPERDATA_REFRESH:
        cmd += ["--superdata-refresh"]
elif MODE == "vvs":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "requests"], check=False)
    cmd = [sys.executable, "vvs_test.py",
           "--base-url", VVS_BASE_URL, "--arms", VVS_ARMS, "--dim", str(VVS_DIM)]
    if VVS_DEBUG:
        cmd += ["--debug"]
    cmd += [VVS_CMD]
else:
    raise SystemExit(f"Unknown MODE={MODE!r} (use 'probe', 'run', or 'vvs').")

if "PASTE_YOUR" in TARGET_API_KEY:
    print("\033[1;33m!! TARGET_API_KEY is still the placeholder - the request will 401. "
          "Edit the CONFIG block.\033[0m")

shown = [("***TOKEN***" if TARGET_API_KEY and c == TARGET_API_KEY else c) for c in cmd]
sh("Executing: " + " ".join(shown))
print("-" * 80)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print("-" * 80)
sh(f"Exit code: {proc.returncode}  |  wall time: {time.time() - t0:.1f}s")
